**Cell 1: Imports and Execution Setup**

In [0]:
from datetime import datetime
import uuid

# Generate a unique execution ID for each pipeline run
execution_id = str(uuid.uuid4())

# Capture the pipeline start time
run_start_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

# Define the project catalog name
catalog_name = "adb_databricks_project_dev"

# Define the metadata schema name
metadata_schema = "metadata"

# Display execution details for tracking and debugging
print("Execution ID:", execution_id)
print("Run Start Time:", run_start_time)

** Auto Entity Discovery**

In [0]:
def discover_entities():

    entities = []

    base_path = "/Volumes/adb_databricks_project_dev/metadata/landing_volume/incoming"

    for item in dbutils.fs.ls(base_path):

        if item.isDir():

            entities.append(
                item.name.replace("/", "")
            )

    return entities


def create_folders(entity_name):

    dbutils.fs.mkdirs(
        f"/Volumes/adb_databricks_project_dev/metadata/landing_volume/archive/{entity_name}"
    )

    dbutils.fs.mkdirs(
        f"/Volumes/adb_databricks_project_dev/metadata/landing_volume/rejected/{entity_name}"
    )


def insert_source_config(entity_name):

    count = spark.sql(f"""

    SELECT COUNT(*)

    FROM adb_databricks_project_dev.metadata.source_config

    WHERE entity_name='{entity_name}'

    """).collect()[0][0]

    if count == 0:

        print(f"Adding : {entity_name}")

        spark.sql(f"""

        INSERT INTO
        adb_databricks_project_dev.metadata.source_config

        VALUES
        (

            uuid(),

            '{entity_name}',

            '{entity_name} File Source',

            'FILE',

            '/Volumes/adb_databricks_project_dev/metadata/landing_volume/incoming/{entity_name}',

            NULL,NULL,NULL,NULL,NULL,NULL,

            'adb_databricks_project_dev',

            'bronze',
            'silver',
            'gold',

            lower('{entity_name}'),

            lower('{entity_name}'),

            lower('{entity_name}_summary'),

            'MASTER',

            'APPEND',

            '_ingestion_time',

            ',',

            'Y',

            NULL,

            '/Volumes/adb_databricks_project_dev/metadata/landing_volume/archive/{entity_name}',

            '/Volumes/adb_databricks_project_dev/metadata/landing_volume/rejected/{entity_name}',

            'Y',

            '{entity_name}ID'

        )

        """)

**insert_source_config**

In [0]:
import uuid

#==========================================================
# Function Name : insert_source_config()
#==========================================================

def insert_source_config(entity_name):

    # Check if entity already exists
    count = spark.sql(f"""

    SELECT COUNT(*)

    FROM adb_databricks_project_dev.metadata.source_config

    WHERE entity_name='{entity_name}'

    """).collect()[0][0]

    if count == 0:

        print(f"Adding : {entity_name}")

        # Generate Source ID
        source_id = str(uuid.uuid4())

        # Insert Metadata
        spark.sql(f"""

        INSERT INTO
        adb_databricks_project_dev.metadata.source_config

        VALUES
        (

            '{source_id}',

            '{entity_name}',

            '{entity_name} File Source',

            'FILE',

            '/Volumes/adb_databricks_project_dev/metadata/landing_volume/incoming/{entity_name}',

            NULL,
            NULL,
            NULL,
            NULL,
            NULL,
            NULL,

            'adb_databricks_project_dev',

            'bronze',
            'silver',
            'gold',

            lower('{entity_name}'),

            lower('{entity_name}'),

            lower('{entity_name}_summary'),

            'MASTER',

            'APPEND',

            '_ingestion_time',

            ',',

            'Y',

            NULL,

            '/Volumes/adb_databricks_project_dev/metadata/landing_volume/archive/{entity_name}',

            '/Volumes/adb_databricks_project_dev/metadata/landing_volume/rejected/{entity_name}',

            'Y',

            '{entity_name}ID'

        )

        """)

        print(f"Inserted Successfully : {entity_name}")

    else:


        print(f"Already Exists : {entity_name}")

In [0]:
#==========================================================
# Auto Onboarding Framework
#==========================================================

entities = discover_entities()

print("Entities Found :")

for entity in entities:

    print(entity)

    create_folders(entity)

    insert_source_config(entity)

**Cell 2: Read Active file in Metadata**

In [0]:
metadata_df = spark.sql(f"""

SELECT *

FROM {catalog_name}.{metadata_schema}.source_config

-- Read only active configurations
WHERE active_flag = 'Y'

-- Currently support only FILE sources
AND source_type = 'FILE'

-- Process entities alphabetically
ORDER BY entity_name

""")

# Display metadata for verification
display(metadata_df)

**Cell 3: JSON Reader**

In [0]:
def read_json_file(file_path):

    # Read a small portion of the file to identify the JSON structure
    file_content = dbutils.fs.head(
        file_path,
        100000
    ).strip()

    # Check if the file contains a JSON Array
    #
    # Example:
    # [
    #   {"OrderID": "1001"},
    #   {"OrderID": "1002"}
    # ]
    if file_content.startswith("["):

        print("JSON Type: ARRAY")

        # Read JSON array using multiLine mode
        return (
            spark.read
            .option("multiLine", "true")
            .json(file_path)
        )

    # Remove empty lines before checking JSON Lines format
    lines = [
        line.strip()
        for line in file_content.splitlines()
        if line.strip()
    ]

    # Check if the file contains JSON Lines (NDJSON)
    #
    # Example:
    # {"OrderID": "1001"}
    # {"OrderID": "1002"}
    if len(lines) > 1 and all(
        line.startswith("{") and line.endswith("}")
        for line in lines
    ):

        print("JSON Type: JSON LINES")

        # Read JSON Lines using single-line mode
        return (
            spark.read
            .option("multiLine", "false")
            .json(file_path)
        )

    # Default case: Single JSON Object
    #
    # Example:
    # {
    #   "OrderID": "1001",
    #   "CustomerID": "C101"
    # }
    print("JSON Type: SINGLE OBJECT")

    # Read single JSON object using multiLine mode
    return (
        spark.read
        .option("multiLine", "true")
        .json(file_path)
    )

**Cell 4: File Reader**

In [0]:
# ==============================================================
# Function Name : read_file_source()
#
# Purpose :
# Reads a source file and automatically detects the
# file format from the file extension.
#
# Supported Formats
#   • CSV
#   • JSON
#   • PARQUET
#
# Called From :
# process_file_source()
#
# Returns :
# Spark DataFrame
#
# Layer :
# Bronze
# ==============================================================


def read_file_source(config, file_path):
    # ----------------------------------------------------------
    # Detect file format from the source file name.
    #
    # Example:
    # order.csv      -> CSV
    # customer.json  -> JSON
    # sales.parquet  -> PARQUET
    # ----------------------------------------------------------

    file_format = file_path.split(".")[-1].upper()

    print("Detected File Format :", file_format)

    # ----------------------------------------------------------
    # Read CSV File
    # ----------------------------------------------------------

    if file_format == "CSV":
        print("Reading CSV :", file_path)

        return (
            spark.read.option(
                "header", "true" if config["header_flag"] == "Y" else "false"
            )
            .option("delimiter", config["delimiter"] if config["delimiter"] else ",")
            .option("inferSchema", "false")
            .csv(file_path)
        )

    # ----------------------------------------------------------
    # Read JSON File
    # ----------------------------------------------------------

    elif file_format == "JSON":
        print("Reading JSON :", file_path)

        return read_json_file(file_path)

    # ----------------------------------------------------------
    # Read PARQUET File
    # ----------------------------------------------------------

    elif file_format == "PARQUET":
        print("Reading PARQUET :", file_path)

        return spark.read.parquet(file_path)

    # ----------------------------------------------------------
    # Unsupported File Format
    # ----------------------------------------------------------

    else:
        raise ValueError(f"Unsupported File Format : {file_format}")

**Cell 6: Check Target Table Exists**

In [0]:

def table_exists(table_name):

    #----------------------------------------------------------
    # Check whether the table exists in the catalog.
    #
    # Example:
    # adb_databricks_project_dev.bronze.order
    #----------------------------------------------------------

    return spark.catalog.tableExists(table_name)

**Cell 7: Write Raw Data to Bronze**

In [0]:
#==============================================================
# Function Name : write_to_bronze()
#
# Purpose :
# Writes the source DataFrame into the Bronze Delta table.
#
# Technical metadata columns are added before writing.
#
# Called From :
# process_file_source()
#
# Called Functions :
# table_exists()
#
# Returns :
# Number of rows loaded.
#
# Layer :
# Bronze
#==============================================================

from pyspark.sql.functions import current_timestamp, lit


def write_to_bronze(
    df,
    config,
    source_name,
    source_path
):

    #----------------------------------------------------------
    # Build fully qualified Bronze table name
    #
    # Example:
    # adb_databricks_project_dev.bronze.order
    #----------------------------------------------------------

    target_table = (
        f"{config['target_catalog']}."
        f"{config['bronze_schema']}."
        f"{config['bronze_table']}"
    )

    #----------------------------------------------------------
    # Detect file format from the source file name.
    #
    # Examples:
    # order.csv      -> CSV
    # customer.json  -> JSON
    # sales.parquet  -> PARQUET
    #----------------------------------------------------------

    file_format = source_name.split(".")[-1].upper()

    #----------------------------------------------------------
    # Add technical metadata columns
    #
    # These columns help with:
    # • Data Lineage
    # • Auditing
    # • Troubleshooting
    # • Monitoring
    #----------------------------------------------------------

    df_bronze = (

        df

        .withColumn(
            "_source_name",
            lit(source_name)
        )

        .withColumn(
            "_source_path",
            lit(source_path)
        )

        .withColumn(
            "_source_type",
            lit(config["source_type"])
        )

        .withColumn(
            "_file_format",
            lit(file_format)
        )

        .withColumn(
            "_entity_name",
            lit(config["entity_name"])
        )

        .withColumn(
            "_source_id",
            lit(config["source_id"])
        )

        .withColumn(
            "_execution_id",
            lit(execution_id)
        )

        .withColumn(
            "_ingestion_time",
            current_timestamp()
        )

    )

    #----------------------------------------------------------
    # Count total records before loading
    #----------------------------------------------------------

    rows_loaded = df_bronze.count()

    if rows_loaded == 0:

        print("No records found.")

        return 0

    try:

        print("--------------------------------------")
        print("Bronze Table :", target_table)
        print("Rows To Load :", rows_loaded)
        print("--------------------------------------")

        #------------------------------------------------------
        # Create Bronze table if it does not exist
        #------------------------------------------------------

        if not table_exists(target_table):

            print("Creating Bronze table...")

            (
                df_bronze.write
                .format("delta")
                .mode("overwrite")
                .saveAsTable(target_table)
            )

            print(f"Bronze table created : {target_table}")

        #------------------------------------------------------
        # Otherwise append data
        #------------------------------------------------------

        else:

            print("Appending records...")

            (
                df_bronze.write
                .format("delta")
                .mode("append")
                .option("mergeSchema", "true")
                .saveAsTable(target_table)
            )

            print(f"Rows appended : {target_table}")

        print(f"Rows Loaded : {rows_loaded}")

        return rows_loaded

    except Exception as e:

        print("--------------------------------------")
        print("Bronze Load Failed")
        print(str(e))
        print("--------------------------------------")

        raise

**Cell 8: SQL Value Helper**

In [0]:
def escape_sql(value):

    # Return an empty string if the value is NULL
    if value is None:
        return ""

    # Escape single quotes to prevent SQL syntax errors
    # Example:
    # O'Brien  -->  O''Brien
    return str(value).replace("'", "''")

**Cell 9: File Log Function**

In [0]:
#==============================================================
# Function Name : insert_file_log()
#
# Purpose :
# Inserts one record into the file processing log table.
#
# File format is automatically detected from the
# source file name.
#
# Called From :
# process_file_source()
#
# Layer :
# Metadata
#==============================================================

def insert_file_log(
    config,
    file_name,
    file_path,
    status,
    error_message=None
):

    #----------------------------------------------------------
    # Build Bronze table name
    #----------------------------------------------------------

    target_table = (
        f"{config['target_catalog']}."
        f"{config['bronze_schema']}."
        f"{config['bronze_table']}"
    )

    #----------------------------------------------------------
    # Detect file format from filename
    #----------------------------------------------------------

    file_format = file_name.split(".")[-1].upper()

    #----------------------------------------------------------
    # Insert processing log
    #----------------------------------------------------------

    spark.sql(f"""

        INSERT INTO
        {config['target_catalog']}.metadata.file_process_log

        VALUES
        (

            '{escape_sql(config["source_id"])}',

            '{escape_sql(config["entity_name"])}',

            '{escape_sql(config["source_type"])}',

            '{escape_sql(file_format)}',

            '{escape_sql(file_name)}',

            '{escape_sql(file_path)}',

            '{escape_sql(target_table)}',

            '{escape_sql(status)}',

            current_timestamp(),

            '{escape_sql(error_message)}'

        )

    """)

**Cell 10: Execution Log Function**

In [0]:
#==============================================================
# Function Name : insert_execution_log()
#
# Purpose :
# Inserts Bronze notebook execution details into execution log.
#==============================================================

def insert_execution_log(
    config,
    source_start_time,
    status,
    files_processed,
    rows_loaded,
    error_message=None
):

    # Bronze layer
    layer = "BRONZE"

    # Entity level execution log
    file_format = "AUTO"

    spark.sql(f"""

        INSERT INTO
        {config['target_catalog']}.metadata.execution_log

        VALUES
        (

            '{execution_id}',

            '{escape_sql(config["entity_name"])}',

            '{layer}',

            '{escape_sql(config["source_type"])}',

            '{file_format}',

            '{source_start_time}',

            current_timestamp(),

            '{escape_sql(status)}',

            {files_processed},

            {rows_loaded},

            '{escape_sql(error_message)}'

        )

    """)

**Cell 11: File Extension Check**

In [0]:
def is_matching_file(file_name, file_format):

    # Convert file name to lowercase
    clean_name = file_name.lower().rstrip("/")

    # Convert file format to uppercase
    file_format = file_format.upper()

    if file_format == "CSV":

        return clean_name.endswith(".csv")

    elif file_format == "JSON":

        return clean_name.endswith(".json")

    elif file_format == "PARQUET":

        return clean_name.endswith(".parquet")

    return False

**Return all file in source folder**

In [0]:
def get_source_files(config):
    """
    Returns all files available in the entity source folder.
    """

    source_path = config["source_path"]

    return dbutils.fs.ls(source_path)

**Archived Function**

In [0]:
#==============================================================
# Function Name : archive_processed_file()
#
# Purpose :
# Moves processed files to Archive or Rejected folder.
#
# SUCCESS -> Archive
# FAILED  -> Rejected
#
# Called From :
# process_file_source()
#
# Layer :
# Bronze
#==============================================================

import os
import uuid

def archive_processed_file(
    source_path,
    target_folder,
    status="SUCCESS"
):

    #----------------------------------------------------------
    # Get source file name
    #----------------------------------------------------------

    file_name = os.path.basename(source_path)

    #----------------------------------------------------------
    # Generate unique archive file name
    #----------------------------------------------------------

    unique_file_name = f"{uuid.uuid4()}_{file_name}"

    destination_path = f"{target_folder}/{unique_file_name}"

    print("======================================")
    print("File Status :", status)
    print("Source      :", source_path)
    print("Destination :", destination_path)

    try:

        dbutils.fs.mv(
            source_path,
            destination_path
        )

        print(f"{status} File Moved Successfully")

    except Exception as e:

        print("File Move Failed")
        print(str(e))

**Cell 12: Process File Source**

In [0]:
#==============================================================
# Function Name : process_file_source()
#
# Purpose :
# Processes all files available in the source folder.
#
# Called From :
# Driver Cell
#
# Called Functions :
#   • get_source_files()
#   • read_file_source()
#   • write_to_bronze()
#   • archive_processed_file()
#   • insert_execution_log()
#
# Layer :
# Bronze
#==============================================================

from datetime import datetime


def process_file_source(config):

    #----------------------------------------------------------
    # Capture source execution start time
    #----------------------------------------------------------

    source_start_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    rows_loaded = 0
    files_processed = 0

    run_status = "SUCCESS"
    error_message = None

    print("===========================================")
    print("Entity        :", config["entity_name"])
    print("Source Type   :", config["source_type"])
    print("Source Folder :", config["source_path"])
    print("===========================================")

    #----------------------------------------------------------
    # Read all files from source folder
    #----------------------------------------------------------

    files = get_source_files(config)

    if len(files) == 0:

        print("No files found.")

        return

    #----------------------------------------------------------
    # Process every file
    #----------------------------------------------------------

    for file in files:

        try:

            print("-------------------------------------------")
            print("Processing File :", file.name)

            #--------------------------------------------------
            # Read file
            #
            # File type is automatically detected inside
            # read_file_source()
            #--------------------------------------------------

            df = read_file_source(

                config=config,

                file_path=file.path

            )

            if df is None:

                print("No data found.")

                continue

            #--------------------------------------------------
            # Write into Bronze
            #--------------------------------------------------

            rows = write_to_bronze(

                df=df,

                config=config,

                source_name=file.name,

                source_path=file.path

            )

            rows_loaded += rows

            files_processed += 1

            #--------------------------------------------------
            # Insert Success File Log
            #--------------------------------------------------

            insert_file_log(

                config=config,

                file_name=file.name,

                file_path=file.path,

                status="SUCCESS"

            )

            #--------------------------------------------------
            # Archive File
            #--------------------------------------------------

            archive_processed_file(

                source_path=file.path,

                target_folder=config["archive_path"],

                status="SUCCESS"

            )

            print("File processed successfully.")

        except Exception as e:

            error_message = str(e)

            run_status = "FAILED"

            print("-------------------------------------------")
            print("Processing Failed")
            print(error_message)
            print("-------------------------------------------")

            #--------------------------------------------------
            # Insert Failed File Log
            #--------------------------------------------------

            insert_file_log(

                config=config,

                file_name=file.name,

                file_path=file.path,

                status="FAILED",

                error_message=error_message

            )

            #--------------------------------------------------
            # Move Failed File
            #--------------------------------------------------

            archive_processed_file(

                source_path=file.path,

                target_folder=config["reject_path"],

                status="FAILED"

            )

    #----------------------------------------------------------
    # Insert Execution Log
    #----------------------------------------------------------

    insert_execution_log(

        config=config,

        source_start_time=source_start_time,

        status=run_status,

        files_processed=files_processed,

        rows_loaded=rows_loaded,

        error_message=error_message

    )

    print("===========================================")
    print("Bronze ingestion completed.")
    print("Files Processed :", files_processed)
    print("Rows Loaded     :", rows_loaded)
    print("Status          :", run_status)
    print("===========================================")

**Cell 14: Main Orchestrator**

In [0]:
#--------------------------------------------------------------
# Convert metadata DataFrame into Python objects.
#
# Every row represents one source configuration.
#--------------------------------------------------------------

metadata_list = metadata_df.collect()

#--------------------------------------------------------------
# Process every metadata configuration one by one.
#--------------------------------------------------------------

for config in metadata_list:

    print("")
    print("######################################################")
    print(f"Starting Entity : {config['entity_name']}")
    print(f"Source Name     : {config['source_name']}")
    print(f"Source Type     : {config['source_type']}")
    print("######################################################")

    #----------------------------------------------------------
    # Process FILE sources
    #
    # Supported Formats
    #   • CSV
    #   • JSON
    #   • PARQUET
    #----------------------------------------------------------

    if config["source_type"] == "FILE":

        process_file_source(config)

    #----------------------------------------------------------
    # Future Source Types
    #
    # DATABASE
    # REST API
    # SAP
    # SFTP
    #
    # These modules will be implemented in future versions.
    #----------------------------------------------------------

    else:

        print(
            f"Source Type '{config['source_type']}' is currently not supported."
        )

print("")
print("======================================================")
print("Bronze Notebook Execution Completed Successfully")
print("======================================================")

Cell 15: Validate Bronze Data

In [0]:
%sql



SELECT *
FROM adb_databricks_project_dev.bronze.`Customer`;

In [0]:
%sql

SELECT *
FROM adb_databricks_project_dev.bronze.`order`;

In [0]:
%sql

SELECT *
FROM adb_databricks_project_dev.bronze.`address`;

In [0]:
%sql

SELECT *
FROM adb_databricks_project_dev.metadata.file_process_log
ORDER BY load_time DESC;

In [0]:
%sql

SELECT *
FROM adb_databricks_project_dev.metadata.execution_log
ORDER BY end_time DESC;